# DefectVision AI - Building/Structural Defect Detection
## Phase 1: Train YOLOv8n on Structural Damage Dataset

This notebook trains a YOLOv8 nano model to detect structural defects in buildings:
- Cracks
- Spalling
- Corrosion
- Exposed rebar

**Run this notebook in Google Colab with GPU runtime.**

1. Go to Runtime > Change runtime type > GPU (T4)
2. Run all cells
3. Download the trained weights at the end

## Step 1: Install Dependencies

In [1]:
!pip install ultralytics roboflow -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.8/91.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 93.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.22.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires google-auth==2.38.0, but you have google-auth 2.47.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyt

## Step 2: Check GPU Availability

In [2]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

PyTorch version: 2.8.0+cu126
CUDA available: False


## Step 3: Download Structural Damage Dataset from Roboflow

This dataset contains images of buildings/structures with annotated damage.

**To get your free Roboflow API key:**
1. Go to [roboflow.com](https://roboflow.com) and create a free account
2. Go to Settings > API Keys
3. Copy your API key and paste it below

In [3]:
from roboflow import Roboflow

# Replace with your Roboflow API key
ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"  # <-- PASTE YOUR KEY HERE

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("university-bswxt").project("crack-bphdr")
version = project.version(2)
dataset = version.download("yolov8")

print(f"\nDataset downloaded to: {dataset.location}")

upload and label your dataset, and get an API KEY here: https://app.roboflow.com/?model=undefined&ref=undefined
loading Roboflow workspace...


RoboflowError: {"error":{"message":"This API key does not exist (or has been revoked).","status":401,"type":"OAuthException","hint":"You may retrieve your API key via the Roboflow Dashboard. Go to Account > Roboflow Keys to retrieve yours.","key":"YOUR_API_KEY_HERE"}}

## Step 4: Explore the Dataset

In [ ]:
import os
import yaml
from pathlib import Path

# Read the data.yaml to see class info
data_yaml_path = os.path.join(dataset.location, "data.yaml")
with open(data_yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print("Dataset configuration:")
print(f"  Classes: {data_config.get('names', 'N/A')}")
print(f"  Number of classes: {data_config.get('nc', 'N/A')}")

# Count images in each split
for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(dataset.location, split, 'images')
    if os.path.exists(img_dir):
        count = len(os.listdir(img_dir))
        print(f"  {split}: {count} images")
    else:
        print(f"  {split}: directory not found")

In [ ]:
# Visualize some sample images
import matplotlib.pyplot as plt
import cv2
import random

train_img_dir = os.path.join(dataset.location, 'train', 'images')
sample_images = random.sample(os.listdir(train_img_dir), min(8, len(os.listdir(train_img_dir))))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, img_name in zip(axes.flatten(), sample_images):
    img = cv2.imread(os.path.join(train_img_dir, img_name))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(img_name[:20], fontsize=8)
    ax.axis('off')
plt.suptitle('Sample Training Images - Building/Structural Defects', fontsize=14)
plt.tight_layout()
plt.show()

## Step 5: Train YOLOv8n Model

Training for 50 epochs. Should take ~20-30 minutes on Colab T4.

In [ ]:
from ultralytics import YOLO

# Load YOLOv8 nano pretrained on COCO (transfer learning)
model = YOLO("yolov8n.pt")

# Train on building/structural defect dataset
results = model.train(
    data=data_yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    name="building_defect_detector",
    patience=10,
    save=True,
    plots=True,
)

## Step 6: Evaluate the Model

In [ ]:
# Validate on the validation set
metrics = model.val()

print(f"\n=== Building Defect Detection Results ===")
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

In [ ]:
# Show training curves
from IPython.display import Image, display

results_dir = Path("runs/detect/building_defect_detector")

for plot_name in ["results.png", "confusion_matrix.png", "val_batch0_pred.png"]:
    plot_path = results_dir / plot_name
    if plot_path.exists():
        print(f"\n--- {plot_name} ---")
        display(Image(filename=str(plot_path), width=800))

## Step 7: Test on Sample Images

In [ ]:
# Run inference on validation images
val_img_dir = os.path.join(dataset.location, 'valid', 'images')
if not os.path.exists(val_img_dir):
    val_img_dir = os.path.join(dataset.location, 'test', 'images')

val_images = os.listdir(val_img_dir)
sample_val = random.sample(val_images, min(6, len(val_images)))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, img_name in zip(axes.flatten(), sample_val):
    img_path = os.path.join(val_img_dir, img_name)
    results = model(img_path, conf=0.25, verbose=False)
    annotated = results[0].plot()
    annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    ax.imshow(annotated)
    ax.set_title(img_name[:25], fontsize=9)
    ax.axis('off')

plt.suptitle('YOLOv8n Predictions - Building/Structural Defects', fontsize=14)
plt.tight_layout()
plt.show()

## Step 8: Download Trained Weights

Download the `best.pt` file and place it in your local project at:
```
defect-vision/models/building_yolo_best.pt
```

In [ ]:
import shutil

# Copy best weights to a convenient location
best_weights = results_dir / "weights" / "best.pt"
if best_weights.exists():
    shutil.copy(best_weights, "building_yolo_best.pt")
    print(f"Best weights saved to: building_yolo_best.pt")
    print(f"File size: {best_weights.stat().st_size / 1e6:.1f} MB")
else:
    alt_path = Path("runs/detect/building_defect_detector/weights/best.pt")
    if alt_path.exists():
        shutil.copy(alt_path, "building_yolo_best.pt")
        print(f"Best weights saved to: building_yolo_best.pt")
    else:
        print("Could not find best.pt - check the runs/ directory")

# Download to your local machine (works in Colab)
try:
    from google.colab import files
    files.download("building_yolo_best.pt")
    print("\nDownload started! Save it to defect-vision/models/building_yolo_best.pt")
except ImportError:
    print("Not running in Colab - copy building_yolo_best.pt manually")